In [1]:
import scanpy as sc
import scipy.io as sio
import pandas as pd

In [2]:
adata = sc.read_h5ad("/rds/general/user/ao225/home/CardiaFinal/Data/Post_Processed/Science/Cardio.h5ad")

In [3]:
adata = adata[adata.obs["disease"].isin(["normal","dilated cardiomyopathy"])]
adata = adata[adata.obs["tissue"] == "heart left ventricle"]
adata = adata[adata.obs["assay"] == "10x 3' v3"]
adata = adata[adata.obs["Primary.Genetic.Diagnosis"].isin(["LMNA", "TTN", "TNNC1", "TNNT2", "RBM20", "PVneg","control"])]

In [5]:
adata.write_h5ad("/rds/general/user/ao225/home/CardiaFinal/Data/Post_Processed/Science/LVCardio.h5ad")

In [5]:
pathway_map = {
    "LMNA":  "chromatin",
    "TTN":   "sarcomere", "TNNC1": "sarcomere", "TNNT2": "sarcomere",
    "RBM20": "splicing",
    "PVneg": "PVneg",
    "control": "control"
}

In [6]:
adata.obs["pathway"] = adata.obs["Primary.Genetic.Diagnosis"].map(pathway_map)

/var/tmp/pbs.3289240.pbs-7/ipykernel_3785851/1988207206.py:1: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adata.obs["pathway"] = adata.obs["Primary.Genetic.Diagnosis"].map(pathway_map)


In [10]:
adata.X

<100528x32383 sparse matrix of type '<class 'numpy.float32'>'
	with 225184014 stored elements in Compressed Sparse Row format>

In [12]:
adata.obs["donor_id"].nunique()

54

In [20]:
output_dir = "/rds/general/user/ao225/home/ext/Science_variant/"
def rogueextracter(adata, name):

    sio.mmwrite(f"{output_dir}expr_{name}.mtx", adata.raw.X.T)
   
    pd.Series(adata.raw.var_names).to_csv(f"{output_dir}genes_{name}.csv", index=False)
    pd.Series(adata.obs_names).to_csv(f"{output_dir}cells_{name}.csv", index=False)
    
    meta_df = adata.obs
    meta_df.to_csv(f"{output_dir}meta_{name}.csv")

In [18]:
rogueextracter(adata, "lv")

In [21]:
control = adata[adata.obs["pathway"] == "control"]
sarcomere  = adata[adata.obs["pathway"] == "sarcomere"]
chromatin     = adata[adata.obs["pathway"] == "chromatin"]
splicing       = adata[adata.obs["pathway"] == "splicing"]
PVneg          = adata[adata.obs["pathway"] == "PVneg"]

In [22]:
rogueextracter(control, "control")
rogueextracter(sarcomere, "sarcomere")
rogueextracter(chromatin, "chromatin")
rogueextracter(splicing, "splicing")
rogueextracter(PVneg, "PVneg")